# FitCheck AI — Model Training (Google Colab)

**Why Colab:** Your Dell has no GPU, so training locally would take ~10 hours. Colab gives you a free T4 GPU and training drops to ~30 minutes.

## Before running this notebook
1. In Colab top menu: **Runtime → Change runtime type → T4 GPU → Save**
2. Zip your `ml/data/processed/` folder on your laptop
3. Upload that zip to your Google Drive (any folder)

Then run each cell in order with Shift+Enter.

## 1. Verify GPU is available

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# If this says CPU, go to Runtime > Change runtime type and pick T4 GPU

## 2. Mount Google Drive and unzip data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT THIS LINE: change 'processed.zip' to whatever you named your zip in Drive
ZIP_PATH = '/content/drive/MyDrive/processed.zip'

import zipfile, os
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/data')

# Check it worked
!ls /content/data/processed/train | head -20
!echo '---'
!ls /content/data/processed/train | wc -l
print('classes above ^')

## 3. Train the model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from pathlib import Path
import time, json

DATA_DIR = Path('/content/data/processed')
MODEL_OUT = Path('/content/fitcheck_mobilenet.pth')
LABELS_OUT = Path('/content/labels.json')

BATCH_SIZE = 64       # GPU can handle larger batches
EPOCHS = 15
LR = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

# Image transforms
train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tfms)
val_ds = datasets.ImageFolder(DATA_DIR / 'val', transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_CLASSES = len(train_ds.classes)
print(f'Classes ({NUM_CLASSES}):', train_ds.classes)

with open(LABELS_OUT, 'w') as f:
    json.dump(train_ds.classes, f)

# Build MobileNetV2 with pretrained weights
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
for param in model.features.parameters():
    param.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_val_acc = 0.0
for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    train_correct, train_total = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_correct += (out.argmax(1) == labels).sum().item()
        train_total += imgs.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += imgs.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train_acc={train_acc:.3f} val_acc={val_acc:.3f} time={time.time()-t0:.0f}s')
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)
        print(f'  ✓ saved best model (val_acc={val_acc:.3f})')

print(f'\nBest val accuracy: {best_val_acc:.3f}')

## 4. Download the trained model to your laptop

In [ ]:
from google.colab import files
files.download('/content/fitcheck_mobilenet.pth')
files.download('/content/labels.json')
# Save these two files. Put them in your project at:
#   backend/app/model.pth
#   backend/app/labels.json